In [0]:
# ── LINE 1: Add widget for parent_run_id (top of notebook) ──
dbutils.widgets.text("parent_run_id", "")
parent_run_id = dbutils.widgets.get("parent_run_id").strip()

In [0]:
# =============================================================================
# Notebook  : 02b_map_07_contacts_attributes
# Location  : /HGI-Lakehouse-Pipeline/03_Silver_Layer/02b_map_07_contacts_attributes
# Spec Ref  : §1.9 contacts_attributes (contact_id, is_paying, is_excluded, mrr)
# Purpose   : Create one attributes row for EVERY contact.
# Run after : map_01 (contacts_all)
# =============================================================================

In [0]:
# CELL 1 ── Load shared config
# In Databricks: each %run must be in its own cell
%run ../config/pipeline_config.py

In [0]:
%run ./_map_common.py

In [0]:
# Load audit logger for error-only logging
%run ../utils/audit_logger

In [0]:
# CELL 2
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").strip().lower()
tenant_id   = tenant_id_from_customer(customer_id)
print(f"=== Map 07: contacts_attributes ===  customer={customer_id}  tenant={tenant_id}")

In [0]:
source_system = "salesforce"
object_name   = "map"
load_type     = "incremental"
initialize_audit_tables()

In [0]:
# CELL 3
create_map_table(
    f"{sv}.contacts_attributes",
    """
        contact_id   STRING NOT NULL,
        tenant       BIGINT,
        is_paying    BOOLEAN,
        is_excluded  BOOLEAN,
        mrr          DOUBLE
    """,
    cluster_by="contact_id"
)

In [0]:
# CELL 4 ── CDF-Aware Incremental MERGE for contacts_attributes (tenant-scoped)
# Reads CDF from contacts_all to find only new/deleted contacts.
# New contacts get default attributes; removed contacts are deleted.
try:
    safe_drop_view(f"{sv}.contacts_attributes")

    # ── VERSION TRACKING: last-processed source version ──
    try:
        props = spark.sql(f"SHOW TBLPROPERTIES {sv}.contacts_attributes ('last_source_cdf_version')").collect()
        val = props[0][1] if props else None
        last_source_ver = int(val) if val and val != '(not specified)' and 'does not have property' not in str(val) else 0
    except Exception:
        last_source_ver = 0

    try:
        source_max_ver = spark.sql(f"SELECT MAX(version) FROM (DESCRIBE HISTORY {sv}.contacts_all)").collect()[0][0] or 0
    except Exception:
        source_max_ver = 0

    print(f"  CDF: source={sv}.contacts_all  last_read_ver={last_source_ver}  current_source_ver={source_max_ver}")

    target_count = spark.sql(f"SELECT COUNT(*) FROM {sv}.contacts_attributes WHERE tenant = {tenant_id}").collect()[0][0]

    if target_count == 0:
        # First run: full MERGE (same as original)
        print(f"  First run — full MERGE from contacts_all")
        spark.sql(f"""
            MERGE INTO {sv}.contacts_attributes AS target
            USING (
                SELECT id AS contact_id, tenant
                FROM {sv}.contacts_all
                WHERE tenant = {tenant_id}
            ) AS source
            ON target.contact_id = source.contact_id AND target.tenant = source.tenant
            WHEN NOT MATCHED THEN INSERT (contact_id, tenant, is_paying, is_excluded, mrr)
                VALUES (source.contact_id, source.tenant, false, false, 0.0)
            WHEN NOT MATCHED BY SOURCE AND target.tenant = {tenant_id} THEN DELETE
        """)
    elif last_source_ver >= source_max_ver:
        print(f"  Source unchanged (ver {source_max_ver}) — skipping")
    else:
        # Incremental: read CDF from contacts_all
        start_ver = last_source_ver + 1
        try:
            cdf_df = (spark.read.format("delta")
                .option("readChangeFeed", "true")
                .option("startingVersion", start_ver)
                .table(f"{sv}.contacts_all")
                .filter(f"tenant = {tenant_id}")
            )

            from pyspark.sql import functions as F
            change_counts = {row['_change_type']: row['count'] for row in cdf_df.groupBy('_change_type').count().collect()}
            print(f"  CDF change types: {change_counts}")

            # INSERT attributes for new contacts
            new_contacts = cdf_df.filter("_change_type = 'insert'").select(F.col("id").alias("contact_id"), "tenant")
            new_count = new_contacts.count()
            if new_count > 0:
                new_contacts.createOrReplaceTempView("new_contacts_cdf")
                spark.sql(f"""
                    MERGE INTO {sv}.contacts_attributes AS target
                    USING new_contacts_cdf AS source
                    ON target.contact_id = source.contact_id AND target.tenant = source.tenant
                    WHEN NOT MATCHED THEN INSERT (contact_id, tenant, is_paying, is_excluded, mrr)
                        VALUES (source.contact_id, source.tenant, false, false, 0.0)
                """)
                print(f"  Inserted {new_count} new contact attributes")

            # DELETE attributes for removed contacts
            deleted_contacts = cdf_df.filter("_change_type = 'delete'").select(F.col("id").alias("contact_id"), "tenant")
            del_count = deleted_contacts.count()
            if del_count > 0:
                deleted_contacts.createOrReplaceTempView("deleted_contacts_cdf")
                spark.sql(f"""
                    MERGE INTO {sv}.contacts_attributes AS target
                    USING deleted_contacts_cdf AS source
                    ON target.contact_id = source.contact_id AND target.tenant = source.tenant
                    WHEN MATCHED THEN DELETE
                """)
                print(f"  Deleted {del_count} removed contact attributes")

            if new_count == 0 and del_count == 0:
                print(f"  CDF: only updates detected — no attribute changes needed")

        except Exception as cdf_err:
            print(f"  CDF read failed ({cdf_err}), falling back to full MERGE")
            spark.sql(f"""
                MERGE INTO {sv}.contacts_attributes AS target
                USING (
                    SELECT id AS contact_id, tenant
                    FROM {sv}.contacts_all
                    WHERE tenant = {tenant_id}
                ) AS source
                ON target.contact_id = source.contact_id AND target.tenant = source.tenant
                WHEN NOT MATCHED THEN INSERT (contact_id, tenant, is_paying, is_excluded, mrr)
                    VALUES (source.contact_id, source.tenant, false, false, 0.0)
                WHEN NOT MATCHED BY SOURCE AND target.tenant = {tenant_id} THEN DELETE
            """)

    # Save version
    spark.sql(f"ALTER TABLE {sv}.contacts_attributes SET TBLPROPERTIES ('last_source_cdf_version' = '{source_max_ver}')")
    print(f"  Saved last_source_cdf_version = {source_max_ver}")

    n = cnt(f"{sv}.contacts_attributes")
    print(f"  contacts_attributes: {n:,} rows")
except Exception as e:
    print(f"\u274c contacts_attributes build failed: {e}")
    log_audit(
        job_name       = "02b_map_07_contacts_attributes",
        customer_id    = customer_id,
        status         = "failure",
        layer          = "silver",
        alert_type     = "FAILURE",
        error_type     = type(e).__name__,
        error_reason   = str(e)[:500],
    )
    raise